# Random Protein Sampling

![Random Protein Sampling](https://proto-bio.github.io/proto-assets/images/tool/random_protein/hero.png)

This notebook demonstrates `run_random_protein_sample`, which generates protein sequence variants by substituting amino acids at masked positions using codon scheme-biased distributions. It covers pre-masked sequences, automatic position selection via masking strategies, and codon scheme comparisons that mirror the amino acid distributions of common degenerate oligonucleotide libraries.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("random_protein")
display_overview("random_protein")
display_docs_section("random_protein", "Background")

## Available tools

In [ ]:
display_available_tools("random_protein")

### `run_random_protein_sample`

`run_random_protein_sample` fills masked positions in protein sequences with amino acids drawn from a codon scheme distribution. Positions marked with `_` are replaced by a randomly sampled amino acid; all other positions are preserved unchanged. The codon scheme controls which amino acids are possible and how frequently each appears: UNIFORM weights all 20 amino acids equally, while library-derived schemes like NNK and NDT reflect the codon redundancy of real degenerate oligonucleotide libraries. The masking strategy system lets you select positions by count, fraction, or index, or you can pre-mask positions manually.

In [ ]:
from proto_tools import RandomProteinSampleInput, RandomProteinSampleConfig, run_random_protein_sample
from proto_tools.transforms.masking import RandomMaskingStrategy

In [ ]:
# Display input docs
display_api_reference("random_protein", "input", "run_random_protein_sample")

# Underscore characters mark positions for random substitution
inputs = RandomProteinSampleInput(sequences=["MK_AY_AKQR"])

In [ ]:
# Display config docs
display_api_reference("random_protein", "config", "run_random_protein_sample")

# Use a masking strategy to automatically select 3 positions to mutate | see docs above for all fields
config = RandomProteinSampleConfig(
    masking_strategy=RandomMaskingStrategy(num_mutations=3),
    seed=42,
)

In [ ]:
# Run the sampling function
wild_type = "MKTLAYAKQR"
result = run_random_protein_sample(
    RandomProteinSampleInput(sequences=[wild_type]),
    config,
)

In [ ]:
# Display output docs
display_api_reference("random_protein", "output", "run_random_protein_sample")

# Show the variant and which positions were mutated
variant = result.sequences[0]
diffs = [i for i, (a, b) in enumerate(zip(wild_type, variant)) if a != b]
print(f"Wild type:         {wild_type}")
print(f"Variant:           {variant}")
print(f"Mutated positions: {diffs}")

# Demonstrate pre-masked input
result_premask = run_random_protein_sample(RandomProteinSampleInput(sequences=["MK_AY_AKQR"]))
print(f"\nPre-masked input:  MK_AY_AKQR")
print(f"Variant:           {result_premask.sequences[0]}")

# Demonstrate codon scheme distributions
print("\nCodon scheme comparison (all-masked sequence):")
for scheme in ["UNIFORM", "NNK", "NDT"]:
    scheme_config = RandomProteinSampleConfig(codon_scheme=scheme, seed=42)
    scheme_result = run_random_protein_sample(
        RandomProteinSampleInput(sequences=["____"]),
        scheme_config,
    )
    print(f"  {scheme:>7s}: {scheme_result.sequences[0]}")

# Demonstrate amino acid exclusions
excluded_config = RandomProteinSampleConfig(excluded_amino_acids=["C"], seed=42)
excluded_result = run_random_protein_sample(
    RandomProteinSampleInput(sequences=["____"]),
    excluded_config,
)
print(f"  no Cys: {excluded_result.sequences[0]}")

## Export Results

Sampling results can be saved to FASTA, TXT, or JSON format for downstream analysis or synthesis ordering.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Generate a batch of 5 variants from the same wild-type sequence
batch_inputs = RandomProteinSampleInput(sequences=["MKTLAYAKQR"] * 5)
batch_config = RandomProteinSampleConfig(masking_strategy=RandomMaskingStrategy(num_mutations=2))
batch_result = run_random_protein_sample(batch_inputs, batch_config)

batch_result.export(name="random_protein_variants", export_path=output_dir, file_format="fasta")
print(f"Exported sequences to {output_dir / 'random_protein_variants.fasta'}")